## 环境准备

In [ ]:
!pip install llama-index-core \
    llama-index-llms-openai \
    llama-index-readers-file \
    llama-index-readers-web

!pip install unstructured[pdf,docx,pptx] \
    paddleocr \
    pillow 

!pip install trafilatura \
    cohere \
    pypdf \
    python-pptx

!pip install sentence-transformers \
    rank_bm25
!pip install llama-index-embeddings-openai
!pip install llama-index-vector-stores-faiss
!pip install llama-index
!pip install faiss-cpu
!pip install fitz
!pip install frontend

LlamaIndex 是一个用于 LLM 应用程序的数据框架，用于注入，结构化，并访问私有或特定领域数据。
llamaindex是目前最适合做RAG的框架。在本质上，LLM（如GPT）为人类和推断出的数据提供了基于自然语言的交互接口。广泛可用的大模型通常在大量公开可用的数据上进行的预训练，包括来自维基百科、邮件列表、书籍和源代码等。
构建在LLM模型之上的应用程序通常需要使用私有或特定领域数据来增强这些模型。不幸的是，这些数据可能分布在不同的应用程序和数据存储中。它们可能存在于API之后、SQL数据库中，或者存在在PDF文件以及幻灯片中。
LlamaIndex应运而生。
主要包含以下模块：
- Data connectors（数据连接器）：用于从各种数据源（如数据库、API、文件系统等）提取和加载数据，以便进行索引和查询。
- Data indexes（数据索引）：负责对文档分chunk，向量化表示，为每个chunk提取元信息（关键词）等，存储已提取的数据并建立索引，以支持高效的信息检索和查询优化。
- Query Engines（引擎）：执行查询处理、语义搜索、RAG（检索增强生成）等任务，确保数据的高效利用和交互。
- Data agents（数据代理）：智能化的数据处理组件，可自动执行任务，如数据转换、合并、增强或基于上下文进行推理。
- Application integrations（应用集成）：提供与外部应用（如LLMs、BI工具、SaaS平台等）的接口，确保数据与现有系统无缝交互。

In [ ]:
llamaindex构建RAG的示例代码：

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = ''
# llamaindex 内嵌的默认模型是gpt-3.5
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
#文档解析
documents = SimpleDirectoryReader('data').load_data()
#构建索引
index = VectorStoreIndex.from_documents(documents)
#构建引擎
query_engine = index.as_query_engine()
#得到结果
response = query_engine.query("收盘价多少")

## Data connectors

上面的示例代码文档读取效果可能不太好，尤其对于表格，图片的信息。llamaindex支持更多的数据读取工具llamahub（https://llamahub.ai/?tab=readers），
以及数据解析云服务llamacloud（https://cloud.llamaindex.ai/）。

下面的代码使用了llamahub中的网页读取器

In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.readers.file import PyMuPDFReader
from llama_index.readers.web import BeautifulSoupWebReader

# 本地文件加载器（扩展支持）
local_loader = SimpleDirectoryReader(
    input_dir="./data",
    required_exts=[".pdf", ".docx", ".pptx", ".epub", ".md"],
    file_extractor={
        ".pdf": PyMuPDFReader(),      # 带文本坐标信息
    }
)

# 网页加载器
web_loader = BeautifulSoupWebReader()

# 示例混合加载
local_docs = local_loader.load_data()
web_docs = web_loader.load_data(urls=[
   "https://cloud.tencent.com/developer/article/2499999?fromSource=gwzcw.9358214.9358214.9358214&utm_medium=cpc&utm_id=gwzcw.9358214.9358214.9358214"  ])
documents = local_docs + web_docs

下面的代码调用llamacloud接口做pdf解析

In [ ]:
# 异步解析
import nest_asyncio
nest_asyncio.apply()

from llama_cloud_services import LlamaParse
parser = LlamaParse(
    api_key="your LLAMA_CLOUD_API_KEY",  
    result_type="markdown",  
    num_workers=3,  # 多文档并行解析
    verbose=True,
    language="ch_sim",  
)

file_extractor = {".pdf": parser}
documents_cloud = SimpleDirectoryReader(
    "./data", file_extractor=file_extractor
).load_data()

## Data Indexes

### Chunking


详见3.1.5 Indexing Chunking，下面给出了llamaindex实现的chunking代码。

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

# 设置 overlap 参数（例如 100 个字符的重叠）
splitter = SentenceSplitter(
    chunk_size=1024,
    chunk_overlap=100,
    paragraph_separator="\n\n",  # 确保段落分隔符不会切分表格
    secondary_chunking_regex="<table>(.+?)</table>",  # 识别表格标签（如果使用HTML格式）
    tokenizer=None  # 可选，使用自定义分词器
)

nodes = splitter.get_nodes_from_documents(documents)

### Indexing

llamaindex内置的数据库检索效率比较低，使用向量数据库能提供更精确的检索结果。下面给出了使用Faiss向量数据库的例子

In [ ]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
# 修改大模型和embedding模型设置
Settings.llm = OpenAI(model="gpt-4o")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

import faiss
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.vector_stores.faiss import FaissVectorStore

# 假设nodes是你已经准备好的节点列表

# 1. 创建FAISS索引
d = 1536  # 对应于text-embedding-3-small的维度
faiss_index = faiss.IndexFlatL2(d)
vector_store = FaissVectorStore(faiss_index=faiss_index)

# 2. 创建存储上下文
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 3. 使用FAISS向量存储创建向量索引
vector_index = VectorStoreIndex(
    nodes,
    storage_context=storage_context
)

# 4. 创建向量检索器
vector_retriever = VectorIndexRetriever(
    index=vector_index,
    similarity_top_k=5  # 设置为你想要的召回数量
)


# 5. 创建查询引擎
query_engine = RetrieverQueryEngine.from_args(
    retriever=vector_retriever
)

# 6. 执行查询
response = query_engine.query("同仁堂安宫牛黄丸的市场价格，中文回答")

如果检索效果不好，还可以查看检索到的内容做debug

In [ ]:
#查看检索的结果
retrieved_nodes = vector_retriever.retrieve("同仁堂安宫牛黄丸的市场价格，中文回答")

# 7. 查看检索到的节点
print(f"共检索到 {len(retrieved_nodes)} 个节点")
for i, node in enumerate(retrieved_nodes):
    print(f"\n节点 {i+1}:")
    print(f"相似度得分: {node.score}")
    print(f"内容: {node.node.text}")
    print("-" * 50)

### Query  Engines

主要功能是处理用户查询，并通过索引和检索机制返回最佳匹配结果，会用到3.1.2 - 3.1.4中介绍的一些RAG技巧
- Query Routing：将query引导到最相关的知识库检索
- Query Rewriting：对query进行多角度重写，去除问题中可能的违法信息
- Query Planning：拆分复杂的问题，规划处解决问题的思路，逐步进行检索

下面代码实现了自定义回答的格式，并实现了带有记忆的聊天

In [ ]:
from llama_index.core.memory import ChatMemoryBuffer

memory = ChatMemoryBuffer.from_defaults(token_limit=5000)

chat_engine = index.as_chat_engine(
    chat_mode="context",
    memory=memory,
    system_prompt=(
        "你是一个聊天机器人"
        " 要基于检索回来的信息回答问题，如果没有检索回来信息，根据你自身的能力回答，不要不回答问题"
    ),
)

response = chat_engine.chat("詹姆斯是谁")
response1 = chat_engine.chat("世运电路2023年上半年实现营业收入多少")
print(response)
print('-'*130)
print(response1)
response1 = chat_engine.chat("你可以告诉我更多吗")
print(response1)

### Data agents

结合大模型（LLM）和工具（工具链、API）来执行复杂任务，如自动化推理、代码执行、任务分解等。
下面的代码实现了全局总结工具和具体问题回复的工具

In [ ]:
import nest_asyncio
nest_asyncio.apply()
from typing import List
from llama_index.core.vector_stores import FilterCondition
from llama_index.core.tools import FunctionTool
from llama_index.core.vector_stores import MetadataFilters


def vector_query(
    query: str, 
    page_numbers: List[str]
) -> str:
    """Perform a vector search over an index.
    
    query (str): the string query to be embedded.
    page_numbers (List[str]): Filter by set of pages. Leave BLANK if we want to perform a vector search
        over all pages. Otherwise, filter by the set of specified pages.
    
    """

    metadata_dicts = [
        {"key": "page_label", "value": p} for p in page_numbers
    ]
    
    query_engine = vector_index.as_query_engine(
        similarity_top_k=5,
        # filters=MetadataFilters.from_dicts(
        #     metadata_dicts,
        #     condition=FilterCondition.OR
        # )
    )
    response = query_engine.query(query)
    return response
    

vector_query_tool = FunctionTool.from_defaults(
    name="vector_tool",
    fn=vector_query
)
from llama_index.core import SummaryIndex
from llama_index.core.tools import QueryEngineTool

summary_index = SummaryIndex(nodes)
summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=True,
)
summary_tool = QueryEngineTool.from_defaults(
    name="summary_tool",
    query_engine=summary_query_engine,
    description=(
        "Useful if you want to get a summary of this article"
    ),
)

In [ ]:
llm=Settings.llm
response3 = llm.predict_and_call(
    [vector_query_tool, summary_tool], 
    "总结一下兴证电子", 
    verbose=True
)
llm=Settings.llm
response3 = llm.predict_and_call(
    [vector_query_tool, summary_tool], 
    "世运电路2023年同比增长", 
    verbose=True
)